In [ ]:
# Make this notebook work from either fine-tuning/ or fine-tuning/live-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


# Lab 02 · Direct Preference Optimization — teach it tone (LIVE)

> **LIVE DEMO LAB.** Resources are suffixed `-live`. The DPO job is submitted but not awaited —
> pivot to the pre-demo lab to show the finished model while this one trains.

A frightened member who just got a cancer diagnosis needs more than a correct answer — a factually-right reply can still land cold, and tone is exactly what members remember. DPO teaches bedside manner from preference pairs: load warm-vs-cold answers (`preferred_output` vs `non_preferred_output`), run a DPO job on `gpt-4o-mini`, and compare base vs DPO on an emotional prompt. The DPO model leads with empathy, then names concrete Sutter next steps. *We didn't add facts — we taught it how Sutter wants members to feel.*

---
## Step 1 — Config & client

In [ ]:
import os, json, time, requests
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential

load_dotenv()

AZURE_OPENAI_ENDPOINT    = os.environ['AZURE_OPENAI_ENDPOINT']
AZURE_OPENAI_API_VERSION = os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
BASE_MODEL               = os.environ.get('BASE_MODEL', 'gpt-4o-mini-2024-07-18')
BASE_DEPLOYMENT          = os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini')
SUBSCRIPTION_ID          = os.environ['AZURE_SUBSCRIPTION_ID']
RESOURCE_GROUP           = os.environ['AZURE_RESOURCE_GROUP']
RESOURCE_NAME            = os.environ['AZURE_RESOURCE_NAME']
TENANT_ID                = os.environ.get('AZURE_TENANT_ID')

SYSTEM_PROMPT = (
    'You are the Sutter Health Member Services assistant. Be warm, empathetic, '
    'and proactive. Always validate the member\'s feelings before solving the '
    'problem, offer multiple concrete next steps, and stay calm.'
)

_cred = DefaultAzureCredential(interactive_browser_tenant_id=TENANT_ID) if TENANT_ID else DefaultAzureCredential()

client = AzureOpenAI(
    azure_endpoint          = AZURE_OPENAI_ENDPOINT,
    azure_ad_token_provider = lambda: _cred.get_token('https://cognitiveservices.azure.com/.default').token,
    api_version             = AZURE_OPENAI_API_VERSION,
)

print(f'Endpoint: {AZURE_OPENAI_ENDPOINT}')
print(f'Base    : {BASE_MODEL}')

---
## Step 2 — Load preference pairs and convert to DPO JSONL

Azure OpenAI DPO format:
```json
{
  "input": { "messages": [{"role":"system",...},{"role":"user",...}] },
  "preferred_output":     [{"role":"assistant","content":"good reply"}],
  "non_preferred_output": [{"role":"assistant","content":"bad reply"}]
}
```

In [ ]:
SRC      = Path('data/sutter_dpo_training_data.json')
DPO_JSONL = Path('data/sutter_dpo.jsonl')

pairs = json.loads(SRC.read_text(encoding='utf-8'))

def _normalize_input(raw_input):
    """Source format: list[{role, content}]. Ensure exactly one system msg
    (ours) followed by the user turn(s)."""
    if isinstance(raw_input, str):
        return [
            { 'role': 'system', 'content': SYSTEM_PROMPT },
            { 'role': 'user',   'content': raw_input },
        ]
    msgs = [m for m in raw_input if m.get('role') != 'system']
    return [{ 'role': 'system', 'content': SYSTEM_PROMPT }, *msgs]

with open(DPO_JSONL, 'w', encoding='utf-8') as f:
    for p in pairs:
        rec = {
            'input': { 'messages': _normalize_input(p['input']) },
            'preferred_output':     [{ 'role': 'assistant', 'content': p['preferred_output'][0]['content'] if isinstance(p['preferred_output'], list) else p['preferred_output'] }],
            'non_preferred_output': [{ 'role': 'assistant', 'content': p['non_preferred_output'][0]['content'] if isinstance(p['non_preferred_output'], list) else p['non_preferred_output'] }],
        }
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

print(f'Wrote {len(pairs)} preference pairs to {DPO_JSONL}')
print('\nSample (first pair):')
with open(DPO_JSONL, 'r', encoding='utf-8') as f:
    print(json.dumps(json.loads(f.readline()), indent=2)[:900] + '...')

---
## Step 3 — BEFORE: base model on emotionally-loaded prompts

In [ ]:
TEST_PROMPTS = [
    'I just got diagnosed with cancer and my whole family is terrified. I have no idea what to do next.',
    'My specialist appointment is three weeks out and my pain is getting worse. I feel like nobody at Sutter cares.',
    'This is the third time I\'ve been billed wrong this year. I\'m exhausted and I\'m about to switch insurance.',
]

print('BEFORE DPO - Base Model')
print('=' * 75)

base_replies = []
for i, q in enumerate(TEST_PROMPTS, 1):
    r = client.chat.completions.create(
        model       = BASE_DEPLOYMENT,
        messages    = [
            { 'role': 'system', 'content': SYSTEM_PROMPT },
            { 'role': 'user',   'content': q },
        ],
        temperature = 0.0,
        max_tokens  = 280,
    )
    a = r.choices[0].message.content.strip()
    base_replies.append(a)
    print(f'\nQ{i}: {q}')
    print(f'A{i}: {a}')

---
## Step 4 — Upload preference JSONL and submit DPO job

**DPO hyperparameters:**
- `beta = 0.1` controls how aggressively the model moves toward preferred (lower = bigger style shift)
- `l2_multiplier = 0.1` keeps the model from drifting too far from the base

In [ ]:
upload = client.files.create(file=open(DPO_JSONL, 'rb'), purpose='fine-tune')
training_file_id = upload.id
print(f'Uploaded: {training_file_id} ({upload.status})')

# Wait for the file to finish processing - DPO jobs reject `pending` files.
print('Waiting for file to be processed...')
for _ in range(60):
    f = client.files.retrieve(training_file_id)
    print(f'  status: {f.status}')
    if f.status == 'processed':
        break
    if f.status in ('error', 'deleted'):
        raise RuntimeError(f'File ended in status {f.status}')
    time.sleep(5)

job = client.fine_tuning.jobs.create(
    training_file = training_file_id,
    model         = BASE_MODEL,
    suffix        = 'sutter-dpo-live',
    seed          = 42,
    method        = {
        'type': 'dpo',
        'dpo': {
            'hyperparameters': {
                'beta': 0.1,
                'l2_multiplier': 0.1,
                'n_epochs': 3,
            }
        }
    },
    extra_body = { 'trainingType': 'GlobalStandard' },
)
job_id = job.id
print(f'\nDPO Job  : {job_id}')
print(f'Status   : {job.status}')

---
## Step 5 — Monitor

In [ ]:
# [LIVE DEMO] Non-blocking monitor.
# This cell does a SINGLE status check and then returns immediately so the
# presenter can pivot to the pre-demo lab while the job cooks server-side.
# Re-run this cell at the end of the demo to confirm completion.
import os, time
from dotenv import load_dotenv

if 'client' not in globals():
    load_dotenv()
    from openai import AzureOpenAI
    from azure.identity import DefaultAzureCredential, get_bearer_token_provider
    _cred = DefaultAzureCredential()
    _tp = get_bearer_token_provider(_cred, 'https://cognitiveservices.azure.com/.default')
    client = AzureOpenAI(
        azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
        api_version=os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview'),
        azure_ad_token_provider=_tp,
    )

job_id = globals().get('job_id')
if not job_id:
    print('  no live job_id in scope. Run the previous (submit) step first.')
else:
    js = client.fine_tuning.jobs.retrieve(job_id)
    print(f'  job_id: {job_id}')
    print(f'  status: {js.status}')
    if js.status == 'succeeded':
        fine_tuned_model = js.fine_tuned_model
        print(f'  fine-tuned model: {fine_tuned_model}')
    elif js.status in ('failed', 'cancelled'):
        print(f'  job ended unexpectedly: {js.status}')
        print(getattr(js, 'error', None))
    else:
        print('  >>> [live demo] not blocking. Pivot to the pre-demo lab now.')
        print('  >>> Refresh Foundry portal -> Build -> Fine-tuning to show the run.')
        print('  >>> Re-run this cell at end of demo for final status.')


---
## Step 6 — DPO loss curve

Lower `train_loss` = model is learning to prefer the good replies.

In [ ]:
final_job = client.fine_tuning.jobs.retrieve(job_id)
if final_job.result_files:
    content = client.files.content(final_job.result_files[0]).read()
    Path('data/sutter_dpo_results.csv').write_bytes(content)
    df = pd.read_csv('data/sutter_dpo_results.csv')

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.set_title('Sutter DPO Training Loss', fontweight='bold')
    if 'train_loss' in df.columns:
        ax.plot(df['step'], df['train_loss'], label='Train Loss', linewidth=1.5)
    if 'full_valid_loss' in df.columns:
        m = df['full_valid_loss'].notna()
        ax.scatter(df.loc[m, 'step'], df.loc[m, 'full_valid_loss'], label='Full Valid Loss', color='red', zorder=5)
    ax.set_xlabel('Step'); ax.set_ylabel('Loss'); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig('data/sutter_dpo_metrics.png', dpi=150); plt.show()
else:
    print('No result file yet.')

---
## Step 7 — Deploy

In [ ]:
# Deploy the DPO model. Uses Developer SKU (free, ephemeral 24h, no quota)
# so it does not contend with the SFT deployment on GlobalStandard.
DPO_DEPLOYMENT_NAME = 'sutter-dpo-live-deployment'

# Explicit model name so this cell works even after a kernel restart
# or if Step 5's variable was lost. Update if you retrain.
fine_tuned_model = globals().get(
    'fine_tuned_model',
    'gpt-4o-mini-2024-07-18.ft-3751051826a54bdbaca296fd325e777a-sutter-dpo',
) or 'gpt-4o-mini-2024-07-18.ft-3751051826a54bdbaca296fd325e777a-sutter-dpo'
print(f'Deploying: {fine_tuned_model}')

auth = _cred.get_token('https://management.azure.com/.default').token
deploy_url = (
    f'https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}'
    f'/resourceGroups/{RESOURCE_GROUP}'
    f'/providers/Microsoft.CognitiveServices/accounts/{RESOURCE_NAME}'
    f'/deployments/{DPO_DEPLOYMENT_NAME}'
)
r = requests.put(
    deploy_url,
    params  = { 'api-version': '2024-10-01' },
    headers = { 'Authorization': f'Bearer {auth}', 'Content-Type': 'application/json' },
    json    = {
        'sku': { 'name': 'Developer', 'capacity': 1 },
        'properties': { 'model': { 'format': 'OpenAI', 'name': fine_tuned_model, 'version': '1' } },
    },
)
print(f'HTTP {r.status_code}: {r.reason}')
if r.status_code not in (200, 201):
    print(json.dumps(r.json(), indent=2))

status_url = deploy_url + '?api-version=2024-10-01'
while True:
    s = requests.get(status_url, headers={ 'Authorization': f'Bearer {auth}' })
    state = s.json().get('properties', {}).get('provisioningState', 'Unknown')
    print(f'[{time.strftime("%H:%M:%S")}] {state}')
    if state == 'Succeeded': break
    if state in ('Failed', 'Canceled'): print(json.dumps(s.json(), indent=2)); break
    time.sleep(20)

---
## Step 8 — AFTER: DPO model on the same prompts

In [ ]:
# Step 8 - AFTER eval. Self-contained: rebuilds any missing state
# (client, prompts, deployment name) so this cell works after a
# kernel restart without re-running every prior cell.
import os, json, sys, time
from pathlib import Path
from dotenv import load_dotenv

print('--- Step 8 starting...', flush=True)

if 'client' not in globals():
    print('  rebuilding client from .env', flush=True)
    load_dotenv()
    from openai import AzureOpenAI
    from azure.identity import DefaultAzureCredential
    _cred = DefaultAzureCredential()
    client = AzureOpenAI(
        azure_endpoint          = os.environ['AZURE_OPENAI_ENDPOINT'],
        azure_ad_token_provider = lambda: _cred.get_token('https://cognitiveservices.azure.com/.default').token,
        api_version             = os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview'),
    )

SYSTEM_PROMPT = globals().get('SYSTEM_PROMPT', (
    'You are the Sutter Health Member Services assistant. Be warm, empathetic, '
    'and proactive. Always validate the member\'s feelings before solving the '
    'problem, offer multiple concrete next steps, and stay calm.'))

TEST_PROMPTS = globals().get('TEST_PROMPTS', [
    'I just got diagnosed with cancer and my whole family is terrified. I have no idea what to do next.',
    'My specialist appointment is three weeks out and my pain is getting worse. I feel like nobody at Sutter cares.',
    "This is the third time I've been billed wrong this year. I'm exhausted and I'm about to switch insurance.",
])

DPO_DEPLOYMENT_NAME = globals().get('DPO_DEPLOYMENT_NAME', 'sutter-dpo-live-deployment')

print(f'  deployment: {DPO_DEPLOYMENT_NAME}', flush=True)
print(f'  prompts:    {len(TEST_PROMPTS)}', flush=True)
print('AFTER DPO - Fine-Tuned Model', flush=True)
print('=' * 75, flush=True)

dpo_replies = []
for i, q in enumerate(TEST_PROMPTS, 1):
    t = time.time()
    print(f'\n[{i}/{len(TEST_PROMPTS)}] asking DPO model... ({q[:60]}...)', flush=True)
    r = client.chat.completions.create(
        model       = DPO_DEPLOYMENT_NAME,
        messages    = [
            { 'role': 'system', 'content': SYSTEM_PROMPT },
            { 'role': 'user',   'content': q },
        ],
        temperature = 0.0,
        max_tokens  = 280,
    )
    a = r.choices[0].message.content.strip()
    dpo_replies.append(a)
    print(f'  ({time.time()-t:.1f}s)', flush=True)
    print(f'Q{i}: {q}', flush=True)
    print(f'A{i}: {a}', flush=True)

# Cache so Step 9 always has fresh results to compare against.
# Preserve any prior 'base' already on disk so re-running Step 8
# doesn't wipe out Step 3's BEFORE replies.
_existing = {}
if Path('dpo_eval_results_live.json').exists():
    try:
        _existing = json.loads(Path('dpo_eval_results_live.json').read_text(encoding='utf-8'))
    except Exception:
        pass
_base_to_cache = globals().get('base_replies') or _existing.get('base', [])
Path('dpo_eval_results_live.json').write_text(json.dumps({
    'prompts': TEST_PROMPTS,
    'base': _base_to_cache,
    'dpo': dpo_replies,
}, indent=2), encoding='utf-8')

print('\n--- Step 8 done. ---', flush=True)


---
## Step 9 — Side-by-side

In [ ]:
# Step 9 - Side-by-side. Self-contained: falls back to the cached
# results from Step 8 if base_replies / dpo_replies aren't in memory.
import json
from pathlib import Path

print('--- Step 9 starting...', flush=True)

_cache_path = Path('dpo_eval_results_live.json')
_need_cache = (
    'base_replies' not in globals() or not globals().get('base_replies')
    or 'dpo_replies' not in globals() or not globals().get('dpo_replies')
)
if _need_cache and _cache_path.exists():
    print('  loading from cache:', _cache_path.resolve(), flush=True)
    _cache = json.loads(_cache_path.read_text(encoding='utf-8'))
    TEST_PROMPTS = _cache.get('prompts', globals().get('TEST_PROMPTS', []))
    base_replies = _cache.get('base', globals().get('base_replies', []))
    dpo_replies  = _cache.get('dpo',  globals().get('dpo_replies',  []))

print('Side-by-Side: Base vs DPO', flush=True)
print('=' * 75, flush=True)
for i, q in enumerate(TEST_PROMPTS):
    print(f'\nQ{i+1}: {q}', flush=True)
    b = base_replies[i] if i < len(base_replies) else '(missing - run Step 3)'
    d = dpo_replies[i]  if i < len(dpo_replies)  else '(missing - run Step 8)'
    print(f'\n  Base : {b}', flush=True)
    print(f'\n  DPO  : {d}', flush=True)
    print('-' * 75, flush=True)

print('\nWhat to highlight in the demo:', flush=True)
print(' - DPO opens with empathy ("I hear you", "that sounds...")', flush=True)
print(' - DPO offers 2-3 concrete next steps, not one boilerplate line', flush=True)
print(' - DPO never tells the member to "just call back later"', flush=True)
print('\n--- Step 9 done. ---', flush=True)


---
## Step 10 — Cleanup

In [ ]:
try:
    client.files.delete(training_file_id); print(f'Deleted file: {training_file_id}')
except Exception as e:
    print(f'Could not delete file: {e}')

auth = _cred.get_token('https://management.azure.com/.default').token
r = requests.delete(deploy_url, params={ 'api-version': '2024-10-01' },
                    headers={ 'Authorization': f'Bearer {auth}' })
print(f'Deployment delete: HTTP {r.status_code}')

print(f'\nDPO model name: {fine_tuned_model}')